# A3 — Efectos heterogéneos: meta-learners y bosque causal (IHDP)

**Curso:** Sistema de Inteligencia Causal · *profesionales*  
**Sesiones:** S13–S15 · **Módulos 2–checkpoint**  
**Lectura monografía:** Cap. **12** Meta-learners · Cap. **13** Bosques causales · Cap. **14** Validación  
**Labs:** L13 · L15  
**Dataset:** `real/ihdp`  
**Técnica:** S/T/X/R-learners, bosque causal, Qini, placebo  
**Alimenta:** **A4** (`artifacts/A3/cate_scores_test.csv`)

### Pregunta de valor (disrupción)

> *Tratar a los de “peor pronóstico” (predicción) no es lo mismo que tratar a los de **mayor efecto** (CATE).*  
> *¿Cambia el ranking de quién recibe la intervención limitada?*

### Objetivos de aprendizaje

1. Estimar **CATE** con meta-learners S/T/X/R.
2. Entrenar un **bosque causal** (o fallback documentado).
3. Validar sin truth en el cuerpo: **Qini** y **placebo**.
4. (Final) PEHE / MAE(τ) con truth solo en evaluación.
5. Elegir learner para A4 (default del curso: **T-learner**).


### Decisión de negocio

> *Si solo predijéramos el outcome, trataríamos a quienes tienen peor pronóstico esperado — sin saber si la intervención les ayuda más que a otros.*  
> *Con el efecto (CATE por meta-learner / bosque causal), la decisión de negocio cambia: identificamos a quién le mueve más la aguja, no a quién le va peor.*

Esta distinción es la que alimenta la política de A4: **priorizar por efecto, no por necesidad aparente**.

## Aviso sobre los datos (transparencia)

Este notebook usa un escenario del **Causal Benchmark Ecosystem**.

Los archivos en `datasets/real/` **no son datos clínicos ni administrativos reales**.  
Lo auténtico de la literatura son **nombres de variables estilo Hill (IHDP), diseño del estudio y órdenes de magnitud de referencia**.  
Los valores numéricos se generan con SEM / fórmulas documentadas para que exista **ground truth** (`truth_*.csv`) usable en evaluación y en el evaluación docente.

| Capa | Archivos | Quién la ve en este notebook |
|------|----------|------------------------------|
| Observables | `observed_*.csv` | **Siempre** (flujo del estudiante) |
| Ground truth | `truth_*.csv` (`y0`, `y1`, `tau`) | **Solo** sección final marcada «evaluación con ground truth / PEHE» |
| Identificación / scorecard | `identification.json`, `scorecard.json` | Lectura guiada; no copiar ciegamente |

Consulta `metadata.json` → campos `what_is_real` / `what_is_synthetic` / `scenario_id`.  
**Política:** no uses `y0`, `y1` ni `tau` fuera de la sección final de evaluación.


## 1. Setup y carga del escenario

Semilla global del curso: `RANDOM_STATE = 42`.  
Splits oficiales del escenario (`observed_train/validation/test`); no re-particionamos.

**Runtime (CPU):** los meta-learners usan RF con `n_estimators=100` (defaults en `lib/cate_utils.py`).  
El bosque causal usa `n_estimators=100`, `cv=3` (ajustes *smoke-friendly*). Para una corrida de aula más completa puedes subir a 200–400 árboles si el tiempo lo permite.


In [ ]:
# Paths relativos al monorepo (kernel-less: funciona con nbconvert/pytest)
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_NOTEBOOK_DIR = Path.cwd()
_CANDIDATES = [
    _NOTEBOOK_DIR.parent,  # .../03_notebooks_caso_estudio if cwd is notebooks/
    _NOTEBOOK_DIR,         # if cwd is already package root
]
for _p in _CANDIDATES:
    if _p is None:
        continue
    if (_p / "lib" / "paths.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        PACKAGE_ROOT = _p
        break
else:
    here = Path.cwd().resolve()
    PACKAGE_ROOT = None
    for parent in [here, *here.parents]:
        if (parent / "lib" / "paths.py").is_file():
            sys.path.insert(0, str(parent))
            PACKAGE_ROOT = parent
            break
    if PACKAGE_ROOT is None:
        raise RuntimeError(
            "No se encontró lib/paths.py. Ejecuta desde "
            "03_notebooks_caso_estudio/ o notebooks/."
        )

from lib.paths import (  # noqa: E402
    RANDOM_STATE,
    SCENARIO_A3,
    SCENARIO_IDS,
    artifacts_dir,
    data_root,
    ensure_dir,
    package_root,
)
from lib.data_io import (  # noqa: E402
    assert_no_truth_columns,
    load_scenario,
)
from lib.cate_utils import (  # noqa: E402
    DEFAULT_METHOD_SELECTED,
    export_a3_artifacts,
    fit_causal_forest,
    fit_meta_learners,
    fit_r_learner,
    fit_s_learner,
    fit_t_learner,
    fit_x_learner,
    placebo_permute_treatment,
    predict_cate_table,
)
from lib.metrics import (  # noqa: E402
    qini_auc,
    qini_curve,
    uplift_curve,
)
from lib.plotting import (  # noqa: E402
    apply_course_style,
    plot_cate_distributions,
    plot_qini,
    plot_uplift_bins,
)

np.random.seed(RANDOM_STATE)
try:
    apply_course_style()
except Exception:
    pass

print("PACKAGE_ROOT :", package_root())
print("DATA_ROOT    :", data_root())
print("RANDOM_STATE :", RANDOM_STATE)
print("SCENARIO_A3  :", SCENARIO_A3, "→", SCENARIO_IDS.get(SCENARIO_A3))

_HAS_ECONML = False
try:
    import econml  # noqa: F401
    _HAS_ECONML = True
except ImportError:
    pass
print("econml available:", _HAS_ECONML)


In [ ]:
# Carga SOLO observables + sidecars (sin truth)
bundle = load_scenario(SCENARIO_A3)
assert_no_truth_columns(bundle.train, context="train")
assert_no_truth_columns(bundle.validation, context="validation")
assert_no_truth_columns(bundle.test, context="test")

meta = bundle.meta
ident = bundle.identification
scorecard = bundle.scorecard

print("scenario_id :", bundle.scenario_id)
print("shapes      : train", bundle.train.shape,
      "| val", bundle.validation.shape,
      "| test", bundle.test.shape)
print("description :", meta.get("description") or meta.get("full_name"))
print("num_units   :", meta.get("num_units") or meta.get("num_samples"))
print("num_covars  :", meta.get("num_covariates"))
print("treatment_r :", meta.get("treatment_ratio"))
print("what_is_real     :", meta.get("what_is_real"))
print("what_is_synthetic:", meta.get("what_is_synthetic"))
print("generator seed (dataset):", meta.get("seed"),
      "| course RANDOM_STATE:", RANDOM_STATE)
# Nota: ate_true vive en metadata; NO lo usamos para estimar en el flujo principal.
print("metadata keys:", sorted(meta.keys()))


In [ ]:
# Roles de columnas y matrices de trabajo
Y_COL = bundle.outcome_column()    # outcome
T_COL = bundle.treatment_column()  # treatment
X_COLS = bundle.covariate_columns()

print(f"Y = {Y_COL!r}  |  T = {T_COL!r}")
print(f"X ({len(X_COLS)}): {X_COLS}")

Y_tr, T_tr, X_tr = bundle.ytx("train")
Y_va, T_va, X_va = bundle.ytx("validation")
Y_te, T_te, X_te = bundle.ytx("test")

print("train treatment rate:", float(T_tr.mean()))
print("train outcome mean  :", float(Y_tr.mean()))
print("val / test n        :", len(Y_va), "/", len(Y_te))
assert list(X_tr.columns) == X_COLS


## 2. EDA causal breve y scorecard de referencia

Antes de CATE: tasa de tratamiento, outcome por brazo y el **piso del curso** en `scorecard.json`  
(baselines ATE y PEHE del T-learner RF del release — solo como ancla de magnitud, no como etiqueta de entrenamiento).


In [ ]:
# EDA de tratamiento y outcome (solo observables)
eda = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n": [len(T_tr), len(T_va), len(T_te)],
    "treat_rate": [float(T_tr.mean()), float(T_va.mean()), float(T_te.mean())],
    "mean_y": [float(Y_tr.mean()), float(Y_va.mean()), float(Y_te.mean())],
    "mean_y_t1": [
        float(Y_tr[T_tr == 1].mean()),
        float(Y_va[T_va == 1].mean()),
        float(Y_te[T_te == 1].mean()),
    ],
    "mean_y_t0": [
        float(Y_tr[T_tr == 0].mean()),
        float(Y_va[T_va == 0].mean()),
        float(Y_te[T_te == 0].mean()),
    ],
})
eda["naive_diff"] = eda["mean_y_t1"] - eda["mean_y_t0"]
print(eda.to_string(index=False))

print("\n--- scorecard baselines (referencia del release) ---")
br = scorecard.get("baseline_results", {})
for name, res in br.items():
    if not isinstance(res, dict):
        continue
    pehe_sc = res.get("pehe")
    ae = res.get("absolute_error")
    ate_e = res.get("ate_estimate")
    print(f"  {name}: ate_hat={ate_e} ae={ae} pehe={pehe_sc}")

print("\nidentification status:", ident.get("status") or ident.get("identifiability"))
print("adjustment set (if any):", ident.get("adjustment_set") or ident.get("conditioning_set"))


## 3. Meta-learners S / T / X / R

Implementaciones didácticas en `lib/cate_utils.py` (sklearn):

| Learner | Idea |
|---------|------|
| **S** | Un solo modelo \(\mu(x,t)\); \(\hat\tau(x)=\mu(x,1)-\mu(x,0)\) |
| **T** | Dos modelos \(\mu_1(x)\), \(\mu_0(x)\); \(\hat\tau=\mu_1-\mu_0\) |
| **X** | Efectos imputados + ponderación por propensity (Künzel et al.) |
| **R** | Residual-on-residual con cross-fitting (Robinson / Nie–Wager simplificado) |

Entrenamos en **train** y predecimos CATE en **validation** y **test** (sin truth).


In [ ]:
# Ajuste individual (auditoría) + batch via fit_meta_learners
# Hiperparámetros moderados: default RF n_estimators=100 en cate_utils

s_model = fit_s_learner(Y_tr, T_tr, X_tr, random_state=RANDOM_STATE)
t_model = fit_t_learner(Y_tr, T_tr, X_tr, random_state=RANDOM_STATE)
x_model = fit_x_learner(Y_tr, T_tr, X_tr, random_state=RANDOM_STATE)
r_model = fit_r_learner(Y_tr, T_tr, X_tr, n_folds=5, random_state=RANDOM_STATE)

print("fitted:", s_model.name, t_model.name, x_model.name, r_model.name)

# Diccionario oficial del notebook (incluye bosque causal en la siguiente sección)
models = {
    s_model.name: s_model,
    t_model.name: t_model,
    x_model.name: x_model,
    r_model.name: r_model,
}

# CATE en test
cate_test = {name: m.predict(X_te) for name, m in models.items()}
summary_rows = []
for name, tau in cate_test.items():
    summary_rows.append({
        "method": name,
        "mean_tau_test": float(np.mean(tau)),
        "std_tau_test": float(np.std(tau)),
        "p10": float(np.quantile(tau, 0.10)),
        "p50": float(np.quantile(tau, 0.50)),
        "p90": float(np.quantile(tau, 0.90)),
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


## 4. Bosque causal (CausalForestDML o fallback)

Si `econml` está instalado usamos `CausalForestDML` (DML + forest honesto).  
Si no, `fit_causal_forest` **documenta** un fallback a T-learner con Random Forest  
(`extra["backend"] = "sklearn_fallback"`). Ambos exponen la misma interfaz `predict(X) → τ̂(x)`.


In [ ]:
# Bosque causal: n_estimators=100, cv=3 (CPU-friendly; ver nota de runtime)
cf_model = fit_causal_forest(
    Y_tr,
    T_tr,
    X_tr,
    n_estimators=100,
    min_samples_leaf=10,
    cv=3,
    random_state=RANDOM_STATE,
)
models[cf_model.name] = cf_model
cate_test[cf_model.name] = cf_model.predict(X_te)

print("causal forest name   :", cf_model.name)
print("backend              :", cf_model.extra.get("backend"))
if cf_model.extra.get("note"):
    print("note                 :", cf_model.extra["note"])
print("mean tau_hat (test)  :", float(np.mean(cate_test[cf_model.name])))
print("models in comparison :", list(models.keys()))


## 5. Comparación visual de distribuciones de CATE

Histogramas de \(\hat\tau(x)\) en el **test** para los cinco métodos.  
Diferencias de forma/escala son esperables (S tiende a suavizar; T/X pueden ser más flexibles; R depende de residuales).


In [ ]:
ART_DIR = ensure_dir(artifacts_dir("A3"))
fig, ax = plt.subplots(figsize=(8.0, 5.0))
plot_cate_distributions(
    cate_test,
    ax=ax,
    title="CATE distributions on test — real/ihdp",
    bins=25,
)
fig.tight_layout()
cate_fig_path = ART_DIR / "cate_comparison.png"
fig.savefig(cate_fig_path, dpi=120)
plt.show()
print("saved:", cate_fig_path)


## 6. Validación offline **sin** ground truth

### 6.1 Curva Qini / uplift

Ordenamos unidades por \(\hat\tau(x)\) decreciente y medimos la ganancia incremental  
en outcomes observados (tratados vs controles). Valores por encima de la línea aleatoria  
sugieren ranking útil — **no** demuestran identificación.

Usamos el **T-learner** (método por defecto del curso para A4) sobre **validation**.


In [ ]:
METHOD_SELECTED = DEFAULT_METHOD_SELECTED  # "t_learner" — contrato A4
assert METHOD_SELECTED in models, (
    f"{METHOD_SELECTED} not in models; got {list(models.keys())}"
)

score_va = models[METHOD_SELECTED].predict(X_va)
curve = qini_curve(Y_va, T_va, score_va, n_grid=80)
auc = qini_auc(curve)
print(f"Qini AUUC-like (val, {METHOD_SELECTED}): {auc:.4f}")

# También Qini para todos los métodos (comparación numérica rápida)
qini_rows = []
for name, m in models.items():
    sc = m.predict(X_va)
    c = qini_curve(Y_va, T_va, sc, n_grid=80)
    qini_rows.append({"method": name, "qini_auc_val": qini_auc(c)})
qini_table = pd.DataFrame(qini_rows).sort_values("qini_auc_val", ascending=False)
print(qini_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.0, 4.5))
plot_qini(curve, ax=ax, title=f"Qini curve (validation) — {METHOD_SELECTED}")
fig.tight_layout()
qini_fig_path = ART_DIR / "qini_plot.png"
fig.savefig(qini_fig_path, dpi=120)
plt.show()
print("saved:", qini_fig_path)

# Uplift por bins (diagnóstico complementario)
uplift_df = uplift_curve(Y_va, T_va, score_va, n_bins=5)
print("\nUplift by score bin (val):")
print(uplift_df.to_string(index=False))
fig2, ax2 = plt.subplots(figsize=(7.0, 4.0))
plot_uplift_bins(uplift_df, ax=ax2, title=f"Uplift bins (val) — {METHOD_SELECTED}")
fig2.tight_layout()
plt.show()


### 6.2 Prueba placebo (permutar tratamiento)

Si permutamos `treatment` y reestimamos el T-learner, el **ATE promedio de τ̂** debería  
colapsar cerca de cero (sanidad del pipeline; no sustituye identificación).  
El evaluación docente exige \(|\mathrm{ATE}_{placebo}| < 0.5\) en la escala IHDP.


In [ ]:
placebo = placebo_permute_treatment(
    Y_tr,
    T_tr,
    X_tr,
    learner="t",
    n_perm=1,
    random_state=RANDOM_STATE,
)
print("learner        :", placebo["learner"])
print("ATE placebo    :", placebo["ate_placebo"])
print("mean |tau_hat| :", placebo["mean_abs_tau"])
print("threshold A3   : |ATE_placebo| < 0.5  →",
      abs(placebo["ate_placebo"]) < 0.5)

# Comparación: ATE "observado" del T-learner real (media de CATE en train)
tau_tr_real = models["t_learner"].predict(X_tr)
print("mean tau_hat (train, real T):", float(np.mean(tau_tr_real)))


## 7. Export de artefactos para A4

Serializamos scores de CATE en **test** con columna `method_selected`  
(por defecto `t_learner`, alineado al plan del curso y al handoff A4→B3).


In [ ]:
# Tabla de scores en test (todas las columnas tau_hat_*)
row_ids = (
    bundle.test["row_id"].to_numpy()
    if "row_id" in bundle.test.columns
    else np.arange(len(bundle.test))
)
scores_df = predict_cate_table(models, X_te, row_ids=row_ids)
scores_df["method_selected"] = METHOD_SELECTED
scores_path = ART_DIR / "cate_scores_test.csv"
scores_df.to_csv(scores_path, index=False)
print("wrote", scores_path, "shape", scores_df.shape)
print("columns:", list(scores_df.columns))
print(scores_df.head(3).to_string(index=False))

# Export de referencia (idempotente; reescribe plots + CSV)
export_info = export_a3_artifacts(
    scenario=SCENARIO_A3,
    out_dir=ART_DIR,
    method_selected=METHOD_SELECTED,
    random_state=RANDOM_STATE,
    n_estimators_cf=100,
    save_plots=True,
)
print("export_a3_artifacts:", {k: export_info[k] for k in (
    "scores_path", "qini_path", "cate_comparison_path",
    "method_selected", "qini_auc_val", "placebo_ate", "cf_backend",
)})


## 8. Discusión guiada (escribir en el entregable)

Completa en ½–1 página (markdown o celda de texto en la entrega):

1. **Desbalance:** la tasa de tratados en train es ~0.43. ¿Cuándo preferirías el **X-learner** frente al **T-learner**?
2. **Qini vs. identificación:** ¿por qué una buena curva Qini **no** demuestra que \(\tau(x)\) esté bien identificado?
3. **Elección para A4:** el curso fija `method_selected = t_learner` por estabilidad y comparabilidad.  
   ¿Coincides? Si no, justifica con Qini / forma de la distribución (sin mirar truth).
4. **Fallback del bosque causal:** si no hay EconML, el backend es un T-learner RF. ¿Qué se pierde respecto a un GRF honesto?

### Respuesta de referencia (modelo de escritura — adaptar con tus números)

> Con tasa de tratamiento ~0.4 el T-learner es razonable; el X-learner gana valor cuando un brazo es pequeño  
> porque impute efectos y recombina con propensity. La Qini solo evalúa ranking bajo outcomes factuales  
> (y supuestos implícitos de solapamiento); no sustituye ignorabilidad. Mantengo `t_learner` para A4  
> por ser el contrato del curso y mostrar dispersión de CATE comparable al scorecard RF.


## Sección de evaluación reservada

La edición pública termina antes de las celdas que requieren `truth`, `y0`, `y1` o `tau` simulados. La evaluación docente se ejecuta en un entorno separado. Continúa la interpretación con datos observados, incertidumbre y supuestos declarados.
